In [13]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [14]:
import sqlite3
from estnltk import Text
from estnltk.taggers import VabamorfAnalyzer
import csv

In [15]:
# sisend
SOURCE_DATA_PATH = "C:/Users/liivas/Downloads/Töö/estnltk_projekt_2025/morphology_conflicts/data/"
TR_SOURCE_DATA_PATH = "C:/Users/liivas/Downloads/Töö/estnltk_projekt_2025/transaktsioonid/source_data/"
TR_DB = "v33_koondkorpus_sentences_verb_pattern_obl_20241002-130310.db"
CONFLICT_DB = "syntax_morphology_conflicts.db"
SENTENCES_DB = "v33_koondkorpus_sentences_sentences_20250220-130121.db"
RESULT = "obj_no_form_homonymy.db"
TR_RESULT = "verb_obj_cases_strict.db"
TR_RESULT2 = "verb_obj_cases_all.db"

In [16]:
morph_analyzer = VabamorfAnalyzer()

In [5]:
# vormihomonüümia (vana)
def has_form_homonymy(phrase_field: str, phrase_root_lemma: str) -> bool:
    phrase_txt = Text(str(phrase_field)).tag_layer("words")
    nom_gen_part_adt = []
    for idx, word in enumerate(phrase_txt.words):
        analysis = morph_analyzer.analyze_token(word.text)
        # huvipakkuv sõna leitakse tabelis oleva phrase_root_lemma alusel
        for a in analysis:
            if a["lemma"] == phrase_root_lemma and a["form"] in ["sg n", "sg g", "sg p", "adt"]:
                # leidsime lemma, mille kääne kuulub vormihomonüümiaohtlike hulka
                for a2 in analysis:
                    if a2["form"] in ["sg n", "sg g", "sg p", "adt"]:
                        nom_gen_part_adt.append(a2["form"])
                break
    if len(nom_gen_part_adt) > 1:
        print(phrase_field, phrase_root_lemma, nom_gen_part_adt)
        return True
    else:
        return False

In [17]:
# (uus)
def has_form_homonymy(word_form: str) -> bool:
    nom_gen_part_adt = []
    analysis = morph_analyzer.analyze_token(word_form)
    # huvipakkuv sõna leitakse tabelis oleva word_form alusel
    for a_idx, a in enumerate(analysis):
        if a["form"] in ["sg n", "sg g", "sg p", "adt"]:
            # leidsime sõna, mille kääne kuulub vormihomonüümiaohtlike hulka
            nom_gen_part_adt.append(a["form"])
    if len(nom_gen_part_adt) > 1:
        #print(phrase_field, word_form, nom_gen_part_adt)
        return True
    else:
        return False

In [18]:
# meetod käände saamiseks feats väljalt
def get_case(feats) -> str:
    cases = ["nom", 
             "gen", 
             "part", 
             "adit", 
             "ill", 
             "el", 
             "all", 
             "term", 
             "abl",
             "kom",
             "ad",
             "es",
             "abes",
             "tr"]
    
    feats_split = feats.split(",")
    case = ""
    # peaks olema 1 kääne per feats, aga järjekord pole fikseeritud
    for idx, feat in enumerate(feats_split):
        if feat in cases:
            case = feat
            break
    return case

In [ ]:
# verbi tegumood (voice)
# prs -> personaal (isikuline)
# imps -> impersonaal (umbisikuline) NB! transaktsioonide andmebaasist on sellised välja jäetud

In [19]:
# verbi aeg (tense)
# pres -> olevik
# past -> minevik

def get_verb_tense(feats: str) -> str|bool:
    if "pres" in feats:
        return "pres"
    elif "past" in feats:
        return "past"
    else:
        return ""

In [20]:
# verbi kõneviis
# käskiv -> imper
# tingiv -> cond
# kindel -> indic
# kaudne -> quot
# möönev -> juss

# muud on partic (mittefiniitsed verbivormid)

def get_verb_mood(feats: str) -> str|bool:
    if "imper" in feats:
        return "imper"
    elif "cond" in feats:
        return "cond"
    elif "indic" in feats:
        return "indic"
    elif "quot" in feats:
        return "quot"
    elif "juss" in feats:
        return "juss"
    else:
        if "partic" in feats:
            return "partic"
        else:
            return ""

### Verbid, obj käänded, sagedused konfliktide andmebaasis

In [19]:
con = sqlite3.connect(f"{SOURCE_DATA_PATH}{CONFLICT_DB}")

con.create_function("has_form_homonymy", 2, has_form_homonymy)

cur = con.cursor()

In [ ]:
cur.execute(f'ATTACH DATABASE "{SOURCE_DATA_PATH}{RESULT}" AS result')

cur.execute("""
DROP TABLE IF EXISTS result.obj_no_form_homonymy
""")

cur.execute(
    """
    CREATE TABLE result.obj_no_form_homonymy
    AS
    SELECT
        *
    FROM
        syntax_morphology_conflicts
    WHERE
        phrase_deprel = "obj"
    AND NOT has_form_homonymy(phrase, phrase_root_lemma)
    LIMIT 500
    """
)

con.close()

#res = cur.fetchall()

aasis Dennis mootorispordibossi mootorispordiboss ['adt', 'sg g', 'sg p']
aasib Leinatamm apsu aps ['adt', 'sg g', 'sg p']
ehk on miski secret aasi secret ['sg n', 'sg p']
abiellume Guyga päeval ristitakse Guyga ['sg g', 'sg n']
major adresseerib Kesküla Kesküla ['sg g', 'sg n']
ta kolhoosi kartulipõllult lahkuma agiteeris kolhoos ['adt', 'sg g', 'sg p']
Praegu agiteerib ta juba ema otsima ema ['sg g', 'sg n', 'sg p']
nikki ei agiteeri teadsa nikk ['adt', 'sg g', 'sg p', 'sg g', 'sg n']
poiss poolõde aias korduvalt seksuaalselt ahistas tõmmates poolõde ['sg n', 'sg p']
õetütar ahistas emakassi emakass ['adt', 'sg g', 'sg p']
ta aias õde seksuaalselt ahistas väitis õde ['sg n', 'sg p']
Ehitustööriist ahistas tõhusalt Jüri Jüri ['sg g', 'sg n']
Mees ahistas vanamemme õhtul territooriumil vanamemm ['adt', 'sg g', 'sg p']
Busemanni ahistab seljavigastus Busemanni ['adt', 'sg g', 'sg p', 'sg g', 'sg n']
ahistas viskejanu viskejanu ['sg g', 'sg n', 'sg p']
Lahte ahistab seljavigastus laht ['

In [21]:
# grupeerime

con = sqlite3.connect(f"{SOURCE_DATA_PATH}{RESULT}")

cur = con.cursor()

cur.execute("""
DROP TABLE IF EXISTS verbs_obj_cases
""")

cur.execute(
    """
    CREATE TABLE verbs_obj_cases
    AS
    SELECT
        verb,
        verb_compound,
        current_case,
        count(*)
    FROM
        obj_no_form_homonymy
    GROUP BY
        verb,
        verb_compound,
        current_case
    ORDER BY
        count(*) DESC
    """
)

con.close()

### Verbid, obj käänded, sagedused transaktsioonide andmebaasis (strict)

In [21]:
con = sqlite3.connect(f"{TR_SOURCE_DATA_PATH}{TR_DB}")

con.create_function("has_form_homonymy", 1, has_form_homonymy)
con.create_function("get_verb_tense", 1, get_verb_tense)
con.create_function("get_case", 1, get_case)
con.create_function("get_verb_mood", 1, get_verb_mood)

cur = con.cursor()
cur.execute(f'ATTACH DATABASE "{SOURCE_DATA_PATH}{TR_RESULT}" AS result')

cur.execute("""
DROP TABLE IF EXISTS result.verbs_obj_cases
""")

cur.execute(
    """
    CREATE TABLE result.verbs_obj_cases
    AS
    SELECT
        tr_head.verb AS verb,
        tr_head.verb_compound AS verb_compound,
        get_verb_tense(tr_head.feats) AS verb_tense,
        get_verb_mood(tr_head.feats) AS verb_mood,
        get_case(tr_row.feats) AS obj_case,
        count(*) AS freq
    FROM
    (
        SELECT
            head_id,
            feats
        FROM
            transaction_row
        WHERE
            deprel = "obj"
        AND NOT
            (get_case(feats) NOT IN ('nom', 'gen', 'part'))
        AND NOT has_form_homonymy(form)
    ) AS tr_row
    INNER JOIN
        transaction_head AS tr_head
    ON 
        tr_row.head_id = tr_head.id
    GROUP BY
        verb,
        verb_compound,
        verb_tense,
        verb_mood,
        obj_case
    ORDER BY
        verb,
        verb_compound,
        freq DESC
    """
)
con.close()

In [15]:
con.close()

Tekitame filtreeritud variandi, kus jätame alles vaid juhud, mis sagedasemad kui 50.

In [23]:
con = sqlite3.connect(f"{SOURCE_DATA_PATH}{TR_RESULT}")

cur = con.cursor()

cur.execute("""
DROP TABLE IF EXISTS verbs_obj_cases_filtered
""")

cur.execute(
    """
    CREATE TABLE verbs_obj_cases_filtered
    AS
    SELECT
        *
    FROM
        verbs_obj_cases
    WHERE
        freq >= 50
    """
)
con.close()

### Käänete osakaalud

In [24]:
con = sqlite3.connect(f"{SOURCE_DATA_PATH}{TR_RESULT}")

cur = con.cursor()

cur.execute("""
DROP TABLE IF EXISTS verbs_obj_case_percentages
""")

cur.execute("""CREATE TABLE verbs_obj_case_percentages AS
SELECT
    verb,
    verb_compound,
    verb_tense,
    verb_mood,

    SUM(freq) AS total,

    100.0 * SUM(CASE WHEN obj_case = 'nom'
                     THEN freq ELSE 0 END)
         / SUM(freq) AS percent_nom,

    100.0 * SUM(CASE WHEN obj_case = 'gen'
                     THEN freq ELSE 0 END)
         / SUM(freq) AS percent_gen,

    100.0 * SUM(CASE WHEN obj_case = 'part'
                     THEN freq ELSE 0 END)
         / SUM(freq) AS percent_par

FROM 
    verbs_obj_cases
GROUP BY
    verb,
    verb_compound,
    verb_tense,
    verb_mood
"""
)

con.close()

In [20]:
con.close()

### Lemmasageduste baasil

In [25]:
con = sqlite3.connect(f"{TR_SOURCE_DATA_PATH}{TR_DB}")

con.create_function("has_form_homonymy", 1, has_form_homonymy)
con.create_function("get_verb_tense", 1, get_verb_tense)
con.create_function("get_case", 1, get_case)
con.create_function("get_verb_mood", 1, get_verb_mood)

cur = con.cursor()
cur.execute(f'ATTACH DATABASE "{SOURCE_DATA_PATH}{TR_RESULT}" AS result')

cur.execute("""
DROP TABLE IF EXISTS result.verbs_obj_cases_lemma
""")

cur.execute(
    """
    CREATE TABLE result.verbs_obj_cases_lemma
    AS
    SELECT
        tr_head.verb AS verb,
        tr_head.verb_compound AS verb_compound,
        get_verb_tense(tr_head.feats) AS verb_tense,
        get_verb_mood(tr_head.feats) AS verb_mood,
        get_case(tr_row.feats) AS obj_case,
        count(DISTINCT tr_row.lemma) AS freq
    FROM
    (
        SELECT
            head_id,
            feats,
            lemma
        FROM
            transaction_row
        WHERE
            deprel = "obj"
        AND
            get_case(feats) IN ('nom', 'gen', 'part')
        AND NOT has_form_homonymy(form)
    ) AS tr_row
    INNER JOIN
        transaction_head AS tr_head
    ON 
        tr_row.head_id = tr_head.id
    GROUP BY
        verb,
        verb_compound,
        verb_tense,
        verb_mood,
        obj_case
    ORDER BY
        verb,
        verb_compound,
        freq DESC
    """
)
con.close()

In [24]:
con = sqlite3.connect(f"{SOURCE_DATA_PATH}{TR_RESULT}")

cur = con.cursor()

cur.execute("""
DROP TABLE IF EXISTS verbs_obj_case_lemma_percentages
""")

cur.execute("""CREATE TABLE verbs_obj_case_lemma_percentages AS
SELECT
    verb,
    verb_compound,
    verb_tense,
    verb_mood,

    SUM(freq) AS total,

    100.0 * SUM(CASE WHEN obj_case = 'nom'
                     THEN freq ELSE 0 END)
         / SUM(freq) AS percent_nom,

    100.0 * SUM(CASE WHEN obj_case = 'gen'
                     THEN freq ELSE 0 END)
         / SUM(freq) AS percent_gen,

    100.0 * SUM(CASE WHEN obj_case = 'part'
                     THEN freq ELSE 0 END)
         / SUM(freq) AS percent_par

FROM 
    verbs_obj_cases_lemma
GROUP BY
    verb,
    verb_compound,
    verb_tense,
    verb_mood
"""
)

con.close()

### Verbid, obj käänded, sagedused transaktsioonide andmebaasis (lenient)

In [7]:
con = sqlite3.connect(f"{TR_SOURCE_DATA_PATH}{TR_DB}")

con.create_function("get_verb_tense", 1, get_verb_tense)
con.create_function("get_case", 1, get_case)
con.create_function("get_verb_mood", 1, get_verb_mood)

cur = con.cursor()
cur.execute(f'ATTACH DATABASE "{SOURCE_DATA_PATH}{TR_RESULT2}" AS result')

cur.execute("""
DROP TABLE IF EXISTS result.verbs_obj_cases_all
""")

cur.execute(
    """
    CREATE TABLE result.verbs_obj_cases_all
    AS
    SELECT
        tr_head.verb AS verb,
        tr_head.verb_compound AS verb_compound,
        get_verb_tense(tr_head.feats) AS verb_tense,
        get_verb_mood(tr_head.feats) AS verb_mood,
        get_case(tr_row.feats) AS obj_case,
        count(*) AS freq
    FROM
    (
        SELECT
            head_id,
            feats
        FROM
            transaction_row
        WHERE
            deprel = "obj"
        AND NOT
            (get_case(feats) NOT IN ('nom', 'gen', 'part'))
    ) AS tr_row
    INNER JOIN
        transaction_head AS tr_head
    ON 
        tr_row.head_id = tr_head.id
    GROUP BY
        verb,
        verb_compound,
        verb_tense,
        verb_mood,
        obj_case
    ORDER BY
        verb,
        verb_compound,
        freq DESC
    """
)
con.close()

In [9]:
con = sqlite3.connect(f"{SOURCE_DATA_PATH}{TR_RESULT2}")

cur = con.cursor()

cur.execute("""
DROP TABLE IF EXISTS verbs_obj_cases_all_filtered
""")

cur.execute(
    """
    CREATE TABLE verbs_obj_cases_all_filtered
    AS
    SELECT
        *
    FROM
        verbs_obj_cases_all
    WHERE
        freq >= 50
    """
)
con.close()

In [10]:
con = sqlite3.connect(f"{SOURCE_DATA_PATH}{TR_RESULT2}")

cur = con.cursor()

cur.execute("""
DROP TABLE IF EXISTS verbs_obj_case_all_percentages
""")

cur.execute("""CREATE TABLE verbs_obj_case_all_percentages AS
SELECT
    verb,
    verb_compound,
    verb_tense,
    verb_mood,

    SUM(freq) AS total,

    100.0 * SUM(CASE WHEN obj_case = 'nom'
                     THEN freq ELSE 0 END)
         / SUM(freq) AS percent_nom,

    100.0 * SUM(CASE WHEN obj_case = 'gen'
                     THEN freq ELSE 0 END)
         / SUM(freq) AS percent_gen,

    100.0 * SUM(CASE WHEN obj_case = 'part'
                     THEN freq ELSE 0 END)
         / SUM(freq) AS percent_par

FROM 
    verbs_obj_cases_all
GROUP BY
    verb,
    verb_compound,
    verb_tense,
    verb_mood
"""
)

con.close()

### Lemmasageduste baasil

In [11]:
con = sqlite3.connect(f"{TR_SOURCE_DATA_PATH}{TR_DB}")

con.create_function("get_verb_tense", 1, get_verb_tense)
con.create_function("get_case", 1, get_case)
con.create_function("get_verb_mood", 1, get_verb_mood)

cur = con.cursor()
cur.execute(f'ATTACH DATABASE "{SOURCE_DATA_PATH}{TR_RESULT2}" AS result')

cur.execute("""
DROP TABLE IF EXISTS result.verbs_obj_cases_all_lemma
""")

cur.execute(
    """
    CREATE TABLE result.verbs_obj_cases_all_lemma
    AS
    SELECT
        tr_head.verb AS verb,
        tr_head.verb_compound AS verb_compound,
        get_verb_tense(tr_head.feats) AS verb_tense,
        get_verb_mood(tr_head.feats) AS verb_mood,
        get_case(tr_row.feats) AS obj_case,
        count(DISTINCT tr_row.lemma) AS freq
    FROM
    (
        SELECT
            head_id,
            feats,
            lemma
        FROM
            transaction_row
        WHERE
            deprel = "obj"
        AND
            get_case(feats) IN ('nom', 'gen', 'part')
    ) AS tr_row
    INNER JOIN
        transaction_head AS tr_head
    ON 
        tr_row.head_id = tr_head.id
    GROUP BY
        verb,
        verb_compound,
        verb_tense,
        verb_mood,
        obj_case
    ORDER BY
        verb,
        verb_compound,
        freq DESC
    """
)
con.close()

In [12]:
con = sqlite3.connect(f"{SOURCE_DATA_PATH}{TR_RESULT2}")

cur = con.cursor()

cur.execute("""
DROP TABLE IF EXISTS verbs_obj_case_lemma_all_percentages
""")

cur.execute("""CREATE TABLE verbs_obj_case_lemma_all_percentages AS
SELECT
    verb,
    verb_compound,
    verb_tense,
    verb_mood,

    SUM(freq) AS total,

    100.0 * SUM(CASE WHEN obj_case = 'nom'
                     THEN freq ELSE 0 END)
         / SUM(freq) AS percent_nom,

    100.0 * SUM(CASE WHEN obj_case = 'gen'
                     THEN freq ELSE 0 END)
         / SUM(freq) AS percent_gen,

    100.0 * SUM(CASE WHEN obj_case = 'part'
                     THEN freq ELSE 0 END)
         / SUM(freq) AS percent_par

FROM 
    verbs_obj_cases_all_lemma
GROUP BY
    verb,
    verb_compound,
    verb_tense,
    verb_mood
"""
)

con.close()